# HW6 — Self-Supervised & Contrastive Representation Learning (STL-10)

**Course:** DATA 266 · **Assignment:** Homework 6 · **Submission:** website URL (your private repo `data266-<SID4>`, tag `hw6`)

Same **ResNet-18 backbone (no pretrained weights)** is used everywhere. We compare three ways of learning a representation under a 10%-label budget on **STL-10**:

- **Part A** — Supervised with limited labels (500 labeled images, end-to-end).
- **Part B** — Rotation self-supervised pretraining → frozen encoder + linear probe.
- **Part C** — SimCLR contrastive pretraining → frozen encoder + linear probe.
- **Part D** — Nearest-neighbour visualization comparing the three encoders.

The three encoders are the "3 different models" the assignment refers to, so no extra hyperparameter-variant model is needed.

> **Runtime note.** STL-10 is ~2.6 GB (downloads on first run) and there are three training regimes. These settings are tuned for a high-end GPU (e.g. RTX 4090): native **96-px** inputs, **all 100k** unlabeled images for rotation, a 50k SimCLR subset, batch 512, and data-loading workers — expect roughly **40–70 min** end-to-end. Every knob is in the **Config** cell; drop `IMG_SIZE`, the unlabeled counts, and batch sizes for a lighter machine. Mixed precision (AMP) is on automatically on CUDA.


## Section 0 — Standing Requirements

### 0.1 Personal parameters
Set `SID4` once; everything else is derived. Per the syllabus, **HW6 uses fixed settings for all students**, so `HP_ID` is *not* applied here. `CLS_A`/`CLS_B` are classification-focus parameters that this assignment does not use (all 10 STL-10 classes are used); they are printed only to satisfy "state them in the first cell."


In [ ]:
# ============================================================
# Section 0.1 — Personal Parameters  (state in first cell)
# ============================================================
SID4 = 2708   # last four digits of SJSU student ID

SEED  = SID4              # torch / numpy / random seed, random_state
SLICE = SID4 % 1000       # start index for deterministic subset selection
HP_ID = SID4 % 6          # experimental arm (NOT used in HW6 — fixed settings)
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

print(f"SID4  = {SID4}")
print(f"SEED  = {SEED}")
print(f"SLICE = {SLICE}")
print(f"HP_ID = {HP_ID}   (HW6: fixed settings for all students -> HP_ID not applied)")
print(f"CLS_A = {CLS_A}, CLS_B = {CLS_B}   (classification-focus params, unused in HW6)")


### 0.2 Reproducibility — dependencies and seeds

In [ ]:
%pip install -q torch torchvision matplotlib numpy


In [ ]:
import os, random, time, json, math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, TensorDataset, Dataset
import torchvision
from torchvision import transforms
from torchvision.datasets import STL10
from torchvision.models import resnet18
import matplotlib.pyplot as plt

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = torch.cuda.is_available()
print("Torch :", torch.__version__, "| torchvision:", torchvision.__version__)
print("Device:", device, "| AMP:", USE_AMP)
if device == "cpu":
    print("WARNING: no GPU found — training will be slow. Enable a GPU runtime and re-run.")

# AMP helpers (new torch.amp API, torch>=2.0)
from torch.amp import autocast, GradScaler
def make_scaler(): return GradScaler("cuda", enabled=USE_AMP)


### Config — all tunable knobs in one place

In [ ]:
# ---- Data / model ----
DATA_ROOT = "./data"
IMG_SIZE  = 96            # STL-10 native resolution (a 4090 handles it easily). Drop to 64 if tight.
NUM_WORKERS = 6           # feeds the GPU. If on WINDOWS you hit a DataLoader pickling error/hang, set 0.
STL_MEAN = (0.4467, 0.4398, 0.4066)
STL_STD  = (0.2603, 0.2566, 0.2713)
CLASSES  = ["airplane","bird","car","cat","deer","dog","horse","monkey","ship","truck"]

# ---- Part A: supervised, limited labels ----
SUP_EPOCHS, SUP_BS, SUP_LR = 15, 64, 1e-3

# ---- Part B: rotation SSL ----
ROT_EPOCHS, ROT_BS, ROT_LR = 15, 512, 1e-3
ROT_UNLABELED_N = None    # None = all 100k unlabeled images (what the assignment asks for)

# ---- Part C: SimCLR ----
SIMCLR_EPOCHS, SIMCLR_BS, SIMCLR_LR = 20, 512, 1e-3
SIMCLR_UNLABELED_N = 50000   # a subset for efficiency (assignment example is 20000; 50k is fine on a 4090)
TEMP, PROJ_DIM = 0.2, 128

# ---- Linear evaluation (Parts B & C) ----
LIN_EPOCHS, LIN_BS, LIN_LR = 20, 256, 1e-3

# Shared DataLoader factory (workers + pinned memory feed the GPU)
def dl(dataset, bs, shuffle=False, drop_last=False):
    return DataLoader(dataset, batch_size=bs, shuffle=shuffle, drop_last=drop_last,
                      num_workers=NUM_WORKERS, pin_memory=(device == "cuda"),
                      persistent_workers=(NUM_WORKERS > 0))

print("IMG_SIZE:", IMG_SIZE, "| rot N: all" if ROT_UNLABELED_N is None else f"| rot N: {ROT_UNLABELED_N}",
      "| simclr N:", SIMCLR_UNLABELED_N, "| workers:", NUM_WORKERS)


### Data — STL-10 splits, the shared 10% labeled subset, and transforms

The **same 500-image labeled subset** (10% of the 5000-image labeled train split, selected with `SEED`) is reused for Part A training and for the Part B/C linear probes, as required.

In [ ]:
norm    = transforms.Normalize(STL_MEAN, STL_STD)
eval_tf = transforms.Compose([transforms.Resize(IMG_SIZE), transforms.ToTensor(), norm])
sup_train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), norm,
])
# SimCLR: the 4 augmentations (random resized crop, horizontal flip, color jitter, grayscale)
simclr_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(), norm,
])

# Datasets (download on first run; ~2.6 GB incl. the 100k unlabeled split)
labeled_eval = STL10(DATA_ROOT, split="train",     transform=eval_tf,      download=True)
labeled_aug  = STL10(DATA_ROOT, split="train",     transform=sup_train_tf, download=False)
test_eval    = STL10(DATA_ROOT, split="test",      transform=eval_tf,      download=True)
unlab_pil    = STL10(DATA_ROOT, split="unlabeled", transform=None,         download=True)   # returns PIL
unlab_tensor = STL10(DATA_ROOT, split="unlabeled", transform=eval_tf,      download=False)  # normalized tensor

# Deterministic 10% labeled subset = 500 images
g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(labeled_eval), generator=g)
LABELED_IDX = perm[:500].tolist()

def pick_unlabeled(n):
    if n is None: return list(range(len(unlab_pil)))
    gg = torch.Generator().manual_seed(SEED)
    return torch.randperm(len(unlab_pil), generator=gg)[:n].tolist()

ROT_IDX    = pick_unlabeled(ROT_UNLABELED_N)
SIMCLR_IDX = pick_unlabeled(SIMCLR_UNLABELED_N)
print(f"labeled subset: {len(LABELED_IDX)} | test: {len(test_eval)} | "
      f"rot unlabeled: {len(ROT_IDX)} | simclr unlabeled: {len(SIMCLR_IDX)}")


### Shared model pieces — ResNet-18 encoder, heads, and the SimCLR NT-Xent loss

In [ ]:
def make_encoder():
    """ResNet-18 (no pretrained weights) with the classification head removed."""
    m = resnet18(weights=None)
    feat_dim = m.fc.in_features      # 512
    m.fc = nn.Identity()
    return m, feat_dim

class Classifier(nn.Module):
    def __init__(self, encoder, feat_dim, n_classes):
        super().__init__()
        self.encoder = encoder
        self.fc = nn.Linear(feat_dim, n_classes)
    def forward(self, x): return self.fc(self.encoder(x))

class SimCLRNet(nn.Module):
    def __init__(self, encoder, feat_dim, proj_dim=128):
        super().__init__()
        self.encoder = encoder
        self.proj = nn.Sequential(nn.Linear(feat_dim, feat_dim), nn.ReLU(inplace=True),
                                  nn.Linear(feat_dim, proj_dim))
    def forward(self, x):
        z = self.proj(self.encoder(x))
        return F.normalize(z, dim=1)

def nt_xent(z, temperature):
    """SimCLR loss. z: [2N, d] L2-normalized, ordered [view1(0..N-1), view2(0..N-1)]."""
    twoN = z.shape[0]; N = twoN // 2
    sim = (z @ z.t()) / temperature                      # cosine-sim / tau
    sim.masked_fill_(torch.eye(twoN, dtype=torch.bool, device=z.device), float("-inf"))
    targets = (torch.arange(twoN, device=z.device) + N) % twoN   # positive of i is i+-N
    return F.cross_entropy(sim, targets)

class RotationDataset(Dataset):
    """Wraps the (normalized-tensor) unlabeled set; returns a randomly rotated image + its label 0..3."""
    def __init__(self, base, indices): self.base, self.indices = base, indices
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        img, _ = self.base[self.indices[i]]
        k = random.randint(0, 3)
        return torch.rot90(img, k, dims=[1, 2]).contiguous(), k

class SimCLRDataset(Dataset):
    """Wraps the (PIL) unlabeled set; returns two independently augmented views."""
    def __init__(self, base, indices, aug): self.base, self.indices, self.aug = base, indices, aug
    def __len__(self): return len(self.indices)
    def __getitem__(self, i):
        img, _ = self.base[self.indices[i]]
        return self.aug(img), self.aug(img)


In [ ]:
@torch.no_grad()
def evaluate_classifier(model):
    model.eval(); correct = tot = 0
    loader = dl(test_eval, 256, shuffle=False)
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with autocast("cuda", enabled=USE_AMP):
            out = model(x)
        correct += (out.argmax(1) == y).sum().item(); tot += y.size(0)
    return correct / tot

@torch.no_grad()
def extract_features(encoder, dataset):
    encoder.eval(); feats, labels = [], []
    loader = dl(dataset, 256, shuffle=False)
    for x, y in loader:
        x = x.to(device)
        with autocast("cuda", enabled=USE_AMP):
            f = encoder(x)
        feats.append(f.float().cpu()); labels.append(y)
    return torch.cat(feats), torch.cat(labels)

def linear_probe(encoder, tag):
    """Freeze encoder, cache features, train a linear classifier on the 500-label subset."""
    for p in encoder.parameters(): p.requires_grad = False
    encoder.eval()
    Xtr, ytr = extract_features(encoder, Subset(labeled_eval, LABELED_IDX))
    Xte, yte = extract_features(encoder, test_eval)
    clf = nn.Linear(Xtr.size(1), 10).to(device)
    opt = torch.optim.Adam(clf.parameters(), lr=LIN_LR)
    tr = DataLoader(TensorDataset(Xtr, ytr), batch_size=LIN_BS, shuffle=True)
    for ep in range(LIN_EPOCHS):
        clf.train()
        for f, y in tr:
            f, y = f.to(device), y.to(device)
            opt.zero_grad(); loss = F.cross_entropy(clf(f), y); loss.backward(); opt.step()
    clf.eval()
    with torch.no_grad():
        acc = (clf(Xte.to(device)).argmax(1).cpu() == yte).float().mean().item()
    print(f"[linear probe: {tag}] test accuracy = {acc:.4f}")
    return acc, clf


## Part A — Supervised with Limited Labels

Train ResNet-18 **end-to-end** on only the 500 labeled images, then report test accuracy on the full 8000-image test set.

In [ ]:
def train_supervised():
    set_seed(SEED)
    enc, fd = make_encoder()
    model = Classifier(enc, fd, 10).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=SUP_LR)
    scaler = make_scaler()
    loader = dl(Subset(labeled_aug, LABELED_IDX), SUP_BS, shuffle=True, drop_last=True)
    for ep in range(SUP_EPOCHS):
        model.train(); lsum = correct = tot = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with autocast("cuda", enabled=USE_AMP):
                out = model(x); loss = F.cross_entropy(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            lsum += loss.item() * x.size(0); tot += x.size(0)
            correct += (out.argmax(1) == y).sum().item()
        print(f"[sup] epoch {ep:2d}  loss {lsum/tot:.4f}  train_acc {correct/tot:.3f}")
    return model

t0 = time.time()
sup_model = train_supervised()
acc_supervised = evaluate_classifier(sup_model)
print(f"\nPart A — Supervised (limited labels) TEST ACCURACY = {acc_supervised:.4f}  "
      f"| {time.time()-t0:.0f}s")
torch.save(sup_model.state_dict(), "hw6_supervised.pt")


## Part B — Rotation Self-Supervised Learning

Pretrain the encoder to classify image rotation (0°/90°/180°/270°) on the unlabeled set, then **freeze** it, drop the rotation head, and train a linear classifier on the same 500-label subset.

In [ ]:
def train_rotation():
    set_seed(SEED)
    enc, fd = make_encoder()
    model = Classifier(enc, fd, 4).to(device)      # 4-way rotation head
    opt = torch.optim.Adam(model.parameters(), lr=ROT_LR)
    scaler = make_scaler()
    loader = dl(RotationDataset(unlab_tensor, ROT_IDX), ROT_BS, shuffle=True, drop_last=True)
    for ep in range(ROT_EPOCHS):
        model.train(); lsum = correct = tot = 0
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            with autocast("cuda", enabled=USE_AMP):
                out = model(x); loss = F.cross_entropy(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            lsum += loss.item() * x.size(0); tot += x.size(0)
            correct += (out.argmax(1) == y).sum().item()
        print(f"[rot] epoch {ep:2d}  loss {lsum/tot:.4f}  rot_acc {correct/tot:.3f}")
    return model.encoder

t0 = time.time()
rot_encoder = train_rotation()
acc_rotation, rot_clf = linear_probe(rot_encoder, "rotation")
print(f"\nPart B — Rotation SSL TEST ACCURACY = {acc_rotation:.4f}  | {time.time()-t0:.0f}s")
torch.save(rot_encoder.state_dict(), "hw6_rotation_encoder.pt")


## Part C — Contrastive Learning (SimCLR-style)

Two augmented views per image (the 4 augmentations above), cosine similarity with temperature **τ = 0.2**, an MLP projection head, NT-Xent loss. Then **freeze** the encoder, drop the projection head, and linear-probe on the same 500-label subset.

In [ ]:
def train_simclr():
    set_seed(SEED)
    enc, fd = make_encoder()
    model = SimCLRNet(enc, fd, PROJ_DIM).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=SIMCLR_LR)
    scaler = make_scaler()
    loader = dl(SimCLRDataset(unlab_pil, SIMCLR_IDX, simclr_tf), SIMCLR_BS, shuffle=True, drop_last=True)
    for ep in range(SIMCLR_EPOCHS):
        model.train(); lsum = tot = 0
        for v1, v2 in loader:
            v1, v2 = v1.to(device), v2.to(device)
            opt.zero_grad()
            with autocast("cuda", enabled=USE_AMP):
                z = torch.cat([model(v1), model(v2)], dim=0)
                loss = nt_xent(z, TEMP)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            lsum += loss.item() * v1.size(0); tot += v1.size(0)
        print(f"[simclr] epoch {ep:2d}  nt_xent loss {lsum/tot:.4f}")
    return model.encoder

t0 = time.time()
simclr_encoder = train_simclr()
acc_simclr, simclr_clf = linear_probe(simclr_encoder, "simclr")
print(f"\nPart C — SimCLR TEST ACCURACY = {acc_simclr:.4f}  | {time.time()-t0:.0f}s")
torch.save(simclr_encoder.state_dict(), "hw6_simclr_encoder.pt")


### Results so far — test accuracy of the three models

In [ ]:
print(f"{'model':<34}{'test acc':>10}")
print(f"{'A. Supervised (10% labels)':<34}{acc_supervised:>10.4f}")
print(f"{'B. Rotation SSL + linear probe':<34}{acc_rotation:>10.4f}")
print(f"{'C. SimCLR + linear probe':<34}{acc_simclr:>10.4f}")


## Part D — Nearest-Neighbour Visualization

For 3 fixed query images (same across all encoders), retrieve the top-5 nearest neighbours by **cosine similarity** in each encoder's embedding space, and show the query, its neighbours, and their class labels.

In [ ]:
encoders = {"Supervised": sup_model.encoder,
            "Rotation SSL": rot_encoder,
            "SimCLR": simclr_encoder}

# L2-normalized test embeddings per encoder (cosine sim == dot product)
embeddings, test_labels = {}, None
for name, enc in encoders.items():
    X, y = extract_features(enc, test_eval)
    embeddings[name] = F.normalize(X, dim=1)
    test_labels = y
print("Embedded test set:", {k: tuple(v.shape) for k, v in embeddings.items()})

# 3 fixed query indices (same for every encoder)
gq = torch.Generator().manual_seed(SEED)
QUERY_IDX = torch.randperm(len(test_eval), generator=gq)[:3].tolist()
print("Query indices:", QUERY_IDX)

def top5_neighbors(name, q):
    sims = embeddings[name] @ embeddings[name][q]
    sims[q] = -2.0                      # exclude the query itself
    return torch.topk(sims, 5).indices.tolist()

inv_norm = transforms.Normalize(mean=[-m/s for m, s in zip(STL_MEAN, STL_STD)],
                                std=[1/s for s in STL_STD])
def show(ax, idx, title):
    img, lbl = test_eval[idx]
    ax.imshow(inv_norm(img).clamp(0, 1).permute(1, 2, 0).numpy())
    ax.set_title(title, fontsize=8); ax.axis("off")


In [ ]:
# One figure per query: rows = encoders, columns = [query | top-5 neighbours]
for q in QUERY_IDX:
    fig, axes = plt.subplots(len(encoders), 6, figsize=(11, 5.5))
    for r, name in enumerate(encoders):
        show(axes[r][0], q, f"QUERY\n({CLASSES[test_labels[q]]})")
        for c, nb in enumerate(top5_neighbors(name, q), start=1):
            show(axes[r][c], nb, CLASSES[test_labels[nb]])
        axes[r][0].set_ylabel(name, fontsize=10, rotation=90, labelpad=12)
        axes[r][0].axis("on"); axes[r][0].set_xticks([]); axes[r][0].set_yticks([])
    fig.suptitle(f"Top-5 cosine neighbours — query #{q} ({CLASSES[test_labels[q]]})", y=1.02)
    fig.tight_layout()
    fig.savefig(f"hw6_nn_query{q}.png", dpi=120, bbox_inches="tight")
    plt.show()
print("Saved -> hw6_nn_query*.png")


### Standing artifacts (Req. 0.3) — write `METRICS.md`

In [ ]:
metrics = {
    "SID4": SID4, "SEED": SEED,
    "backbone": "resnet18 (no pretrained weights)",
    "img_size": IMG_SIZE,
    "labeled_subset": len(LABELED_IDX),
    "rot_unlabeled": len(ROT_IDX), "simclr_unlabeled": len(SIMCLR_IDX),
    "simclr": {"temperature": TEMP, "proj_dim": PROJ_DIM},
    "test_accuracy": {
        "A_supervised": round(acc_supervised, 4),
        "B_rotation":   round(acc_rotation, 4),
        "C_simclr":     round(acc_simclr, 4),
    },
    "query_indices": QUERY_IDX,
}
with open("metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

lines = [
    "# METRICS - HW6 Self-Supervised / Contrastive (STL-10)\n",
    f"- SID4={SID4}, SEED={SEED}",
    f"- Backbone: resnet18 (no pretrained) | img_size={IMG_SIZE}",
    f"- Labeled subset: {len(LABELED_IDX)} | rotation unlabeled: {len(ROT_IDX)} | "
    f"simclr unlabeled: {len(SIMCLR_IDX)} | tau={TEMP}\n",
    "## Test accuracy (full 8000-image test set)\n",
    "| part | model | test accuracy |",
    "|---|---|---|",
    f"| A | Supervised (10% labels, end-to-end) | {acc_supervised:.4f} |",
    f"| B | Rotation SSL + linear probe | {acc_rotation:.4f} |",
    f"| C | SimCLR + linear probe | {acc_simclr:.4f} |",
]
with open("METRICS.md", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print("\n".join(lines))
print("\nSaved -> METRICS.md, metrics.json")


## Comparison & Discussion (Part D write-up)

**Test accuracy (full 8000-image test set): SimCLR 0.5838 ≫ Supervised 0.4000 ≈ Rotation 0.3935.**

**Which model performed best, and why.** SimCLR won by a wide margin — about **+18 points** over the next model. Its contrastive objective pulls two augmentations of the same image together and pushes all other images apart, which directly shapes an embedding space where same-class images cluster — exactly what a linear classifier (and cosine nearest-neighbour search) needs. Pretrained on 50k unlabeled images, its frozen features already encode object semantics, so the 500 labels only have to fit a 10-way linear layer.

**Why the others performed the way they did.** Rotation SSL (0.3935) did **not** beat the from-scratch supervised model (0.4000) here — the two are effectively tied. Predicting 0°/90°/180°/270° rewards features sensitive to canonical pose and global layout, which transfer only partially to object-class identity; with the encoder frozen, those features weren't rich enough to out-probe even a data-starved supervised net. The supervised model is limited by data rather than pretext mismatch: 500 images across 10 classes is very little to train a full ResNet-18 end-to-end, so it overfits and generalizes modestly.

**What the nearest-neighbour visualizations revealed.** The figures track the accuracy ranking and expose *what kind* of similarity each encoder learned:
- **Ship query (#526):** SimCLR returned 5/5 ships; Rotation 3/5 (two airplanes — both share sky/horizon and an elongated metallic body); Supervised only 2/5, mixing in airplanes and a car — matching on low-level cues (water, sky, perspective) rather than object identity.
- **Cat query (#6237):** SimCLR 4/5 cats; Rotation 2/5 (plus a dog and two birds); Supervised was the noisiest (deer, dog, bird, cat, dog), keying on orange/brown fur and generic animal shape rather than "cat."
- **Car query (#6399):** Rotation actually nailed this one (5/5 cars), SimCLR 4/5, Supervised 2/5 (trucks and a ship — dark vehicle silhouettes). This is revealing: rotation's pose-sensitivity helps rigid, canonically-oriented man-made objects (cars) but fails on deformable animals (its cat/bird confusion), which is exactly why its *average* trails SimCLR even though it can match or beat it on individual rigid-object classes.

So SimCLR's embedding is the most **semantically** organized (class-coherent neighbours across all three queries), rotation's is organized by **pose/appearance** (great for cars, weak for animals), and the supervised encoder's is the least class-coherent — neighbours share texture/colour/silhouette more than identity.

**What could improve each model.** *Supervised:* more labels, stronger augmentation/regularization, or initializing from one of the SSL encoders instead of scratch. *Rotation:* a harder or combined pretext task (jigsaw, exemplar, or adding rotation to a contrastive loss) so it captures semantics rather than pose — it already used all 100k images, so more pretext data alone won't help much. *SimCLR:* larger batch (more negatives), more pretraining epochs, a LARS optimizer with a cosine schedule, and fine-tuning (not just linear-probing) the encoder — it has the most headroom and was already the best.

## Files produced

| File | For |
|---|---|
| `hw6_supervised.pt` | Part A model |
| `hw6_rotation_encoder.pt` | Part B encoder |
| `hw6_simclr_encoder.pt` | Part C encoder |
| `hw6_nn_query*.png` | Part D nearest-neighbour figures |
| `METRICS.md`, `metrics.json` | Req. 0.3 metrics |

**Repo checklist (Req. 0.3):** executed `.ipynb`, `METRICS.md`, `RUN_LOG.txt`, `AI_USE.md`, and the three checkpoints in your private repo `data266-<SID4>`, tagged **`hw6`**. Commit **incrementally** — a single deadline commit is capped at half credit on implementation.
